# Ejercicio 5: Preprocesamiento

### Leandro Bravo

# Paso 1: Remover Caracter Especial

In [83]:
import re

def remove_special_characters(text):
    cleaned_text = re.sub(r'[^a-zA-Z0-9áéíóúÁÉÍÓÚñÑ\s]', '', text)
    return cleaned_text


### 1.1 Leer Corpus

In [84]:
import os
path = os.path.normpath(r'C:\Users\leand\Desktop\ir2026\ir-2026-BLeandro\ir26a-main\data')
corpus_dir = os.path.join(path, '01_corpus_turismo_500.txt')

with open(corpus_dir, 'r', encoding='utf-8') as f:
    corpus = f.read()
print(corpus[:100])  

Otavalo es conocido por su mercado indígena y su artesanía Perfecto para rafting.
La Laguna Quilotoa


In [85]:
import pandas as pd
df_corpus = pd.DataFrame({'corpus': [corpus]})
df_corpus

,corpus
0,Otavalo es conocido por su mercado indígena y ...


### 1.2 Limpiar

In [91]:
df_corpus['corpus'] = df_corpus['corpus'].apply(remove_special_characters)

df_corpus

,corpus
0,Otavalo es conocido por su mercado indígena y ...


# Paso 2: Tokenizar

In [92]:
!pip install nltk

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

import string

def preprocess_text_tokenizer(text, language='spanish'):
    language = language.lower()
    if language not in {'spanish', 'english'}:
        raise ValueError("Language must be 'spanish' or 'english'")
    
    tokens = word_tokenize(text, language=language)
    stop_words = set(stopwords.words(language))
    
    tokens = [
        token.lower() 
        for token in tokens 
        if token.lower() not in stop_words and token.isalpha()
    ]
    
    return tokens



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: C:\Users\leand\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\leand\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\leand\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\leand\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [93]:
def preprocess_text_tokenizer_without_stopwords(text, language='spanish'):
    tokens = word_tokenize(text, language=language)
    tokens = [token.lower() for token in tokens if token.isalpha()]
    return tokens

### 2.1 Tokenizar corpus

In [100]:
df_corpus['tokens'] = df_corpus['corpus'].apply(
    lambda x: preprocess_text_tokenizer_without_stopwords(x, language='spanish')
)
df_corpus

,corpus,tokens
0,Otavalo es conocido por su mercado indígena y ...,"[otavalo, es, conocido, por, su, mercado, indí..."


# Paso 3: Lemmatization/Stemming

In [61]:
# Stemming
from nltk.stem import SnowballStemmer

def stemming_tokens(tokens, language='spanish', return_text=False):
    language = language.lower()
    
    if language not in {'spanish', 'english'}:
        raise ValueError("Language must be 'spanish' or 'english'")
    
    stemmer = SnowballStemmer(language)
    
    stemmed_tokens = [
        stemmer.stem(token)
        for token in tokens
    ]
    
    if return_text:
        return stemmed_tokens, " ".join(stemmed_tokens)
    
    return stemmed_tokens

In [57]:
# Lemmatization
from nltk.stem import SnowballStemmer

def lemmatization_tokens(tokens, language='spanish', return_text=False):
    if language not in {'spanish', 'english'}:
        raise ValueError("Language must be 'spanish' or 'english'")
    
    stemmer = SnowballStemmer(language)
    
    lemmas = [stemmer.stem(token) for token in tokens]
    
    if return_text:
        return lemmas, " ".join(lemmas)
    
    return lemmas

### 3.1 Stemming corpus

In [103]:
df_corpus['result'] = df_corpus['tokens'].apply(lambda x: " ".join(stemming_tokens(x, language='spanish')))
df_corpus

,corpus,tokens,result
0,Otavalo es conocido por su mercado indígena y ...,"[otavalo, es, conocido, por, su, mercado, indí...",otaval es conoc por su merc indigen y su artes...


# Paso 4: TF-IDF

In [110]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = TfidfVectorizer()

tfidf = vectorizer.fit_transform(df_corpus['result'])

query = "conocido destino turístico en España"
query_tfidf = vectorizer.transform([query])

df_corpus['dist'] = cosine_similarity(tfidf, query_tfidf).flatten()
df_corpus.sort_values(by='dist', ascending=False)

,corpus,tokens,result,dist
0,Otavalo es conocido por su mercado indígena y ...,"[otavalo, es, conocido, por, su, mercado, indí...",otaval es conoc por su merc indigen y su artes...,0.034473
